# FLEX Simulation Comparison

## Validating the FLEX Replacement Level Methodology

This notebook compares the FLEX (greedy-by-ADP) approach to alternative draft simulation strategies:

1. **FLEX (ADP-Greedy)**: Our current methodology - simulates draft by drafting best available player by ADP
2. **RB-Heavy**: Prioritizes RBs in early rounds (2010s draft strategy)
3. **WR-Heavy**: Prioritizes WRs early (modern "zero-RB" adjacent)
4. **Zero-RB**: Avoids RBs until mid-rounds
5. **Manual/Subjective**: Hand-picked "typical" roster construction

We'll show how each strategy affects:
- Replacement level identification (QB12, RB28, WR32, TE12)
- Scarcity multipliers (VOR weights)
- Overall draft value distribution

**Conclusion**: FLEX is the most defensible because it uses consensus market value (ADP) without subjective bias.

In [ ]:
# Import libraries
import duckdb
import pandas as pd
import plotly.express as px

# Connect to warehouse
conn = duckdb.connect('../data/warehouse.duckdb', read_only=True)

print("✅ Libraries loaded and database connected")

## 1. Load Preseason Rankings (ADP Data)

First, we load the preseason ADP rankings that serve as input to all draft simulations.

In [ ]:
# Load preseason rankings
preseason = conn.execute("""
    SELECT 
        player_id,
        player_name,
        position,
        team,
        preseason_rank_overall,
        preseason_rank_position,
        preseason_adp
    FROM stg_preseason_rankings
    WHERE preseason_adp IS NOT NULL
    ORDER BY preseason_adp
""").df()

print(f"Loaded {len(preseason)} players with ADP data")
print("\nPosition breakdown:")
print(preseason['position'].value_counts())
preseason.head(20)

## 2. Simulate Different Draft Strategies

We'll simulate a 10-team, 14-round draft (140 picks) using different strategies:

### Strategy Definitions:
- **FLEX (ADP)**: Draft best available by ADP (our current method)
- **RB-Heavy**: Force 2 RBs in first 3 rounds, then BPA
- **WR-Heavy**: Force 2 WRs in first 3 rounds, then BPA  
- **Zero-RB**: No RBs until round 5+
- **Manual**: Typical 2RB/2WR/1TE/1QB split by round 6

In [ ]:
def simulate_draft(strategy='flex', num_teams=10, rounds=14):
    """
    Simulate a fantasy draft using different strategies.
    Returns DataFrame of drafted players with pick number and team.
    """
    draft_pool = preseason.copy()
    draft_results = []
    pick_no = 0
    
    for round_num in range(1, rounds + 1):
        for team in range(1, num_teams + 1):
            pick_no += 1
            
            # Determine position requirement based on strategy
            if strategy == 'flex':
                # Pure BPA by ADP
                required_pos = None
            elif strategy == 'rb_heavy':
                # Force RB in rounds 1-2, then BPA
                required_pos = 'RB' if round_num <= 2 else None
            elif strategy == 'wr_heavy':
                # Force WR in rounds 1-2, then BPA
                required_pos = 'WR' if round_num <= 2 else None
            elif strategy == 'zero_rb':
                # Avoid RB until round 5
                required_pos = 'NOT_RB' if round_num < 5 else None
            elif strategy == 'manual':
                # Typical roster construction
                if round_num <= 2:
                    required_pos = 'RB'
                elif round_num <= 4:
                    required_pos = 'WR'
                elif round_num == 5:
                    required_pos = 'TE'
                elif round_num == 6:
                    required_pos = 'QB'
                else:
                    required_pos = None
            
            # Draft player
            if required_pos == 'NOT_RB':
                available = draft_pool[draft_pool['position'] != 'RB']
            elif required_pos:
                available = draft_pool[draft_pool['position'] == required_pos]
            else:
                available = draft_pool
            
            if len(available) > 0:
                # Draft best available (lowest ADP)
                pick = available.iloc[0]
                draft_results.append({
                    'pick_no': pick_no,
                    'round': round_num,
                    'team': team,
                    'player_id': pick['player_id'],
                    'player_name': pick['player_name'],
                    'position': pick['position'],
                    'preseason_adp': pick['preseason_adp'],
                    'strategy': strategy
                })
                # Remove from pool
                draft_pool = draft_pool[draft_pool['player_id'] != pick['player_id']]
    
    return pd.DataFrame(draft_results)

# Run all simulations
strategies = {
    'FLEX (ADP)': 'flex',
    'RB-Heavy': 'rb_heavy',
    'WR-Heavy': 'wr_heavy',
    'Zero-RB': 'zero_rb',
    'Manual': 'manual'
}

all_drafts = []
for name, strat in strategies.items():
    print(f"Simulating {name}...")
    draft = simulate_draft(strategy=strat)
    draft['strategy_name'] = name
    all_drafts.append(draft)

combined_drafts = pd.concat(all_drafts, ignore_index=True)
print(f"\n✅ Simulated {len(strategies)} draft strategies")
print(f"Total picks: {len(combined_drafts)}")

## 3. Identify Replacement Levels for Each Strategy

The FLEX position is crucial - it determines the replacement level baseline. Different draft strategies create different player pools.

In [ ]:
def find_replacement_levels(draft_df, num_teams=10):
    """
    Find replacement level players for each position based on draft results.
    Replacement = First player NOT drafted (1st bench player at each position)
    
    Standard rosters: 1 QB, 2 RB, 2 WR, 1 TE, 1 FLEX = 28 RBs, 32 WRs
    """
    replacement_ranks = {
        'QB': 12,   # 10 starters + 2 bench
        'RB': 28,   # 20 starters (2 RB + ~3 FLEX) + 8 bench
        'WR': 32,   # 20 starters (2 WR + ~7 FLEX) + 12 bench
        'TE': 12    # 10 starters + 2 bench
    }
    
    replacements = []
    for pos in ['QB', 'RB', 'WR', 'TE']:
        pos_drafted = draft_df[draft_df['position'] == pos].sort_values('pick_no')
        if len(pos_drafted) >= replacement_ranks[pos]:
            replacement = pos_drafted.iloc[replacement_ranks[pos] - 1]
            replacements.append({
                'position': pos,
                'replacement_rank': replacement_ranks[pos],
                'player_name': replacement['player_name'],
                'pick_no': replacement['pick_no'],
                'round': replacement['round'],
                'preseason_adp': replacement['preseason_adp']
            })
    
    return pd.DataFrame(replacements)

# Find replacement levels for each strategy
replacement_comparison = []
for name, strat in strategies.items():
    draft = combined_drafts[combined_drafts['strategy_name'] == name]
    replacements = find_replacement_levels(draft)
    replacements['strategy'] = name
    replacement_comparison.append(replacements)

replacement_df = pd.concat(replacement_comparison, ignore_index=True)

# Pivot for easy comparison
replacement_pivot = replacement_df.pivot(
    index='position',
    columns='strategy',
    values='player_name'
)

print("Replacement Level Players by Strategy:\n")
print(replacement_pivot)
print("\n" + "="*80 + "\n")

# Show pick numbers
pick_pivot = replacement_df.pivot(
    index='position',
    columns='strategy',
    values='pick_no'
)
print("Pick Number of Replacement Level:\n")
print(pick_pivot)

## 4. Visualize Replacement Level Differences

Let's see how each strategy affects when replacement-level players are drafted.

In [ ]:
# Create grouped bar chart of replacement pick numbers
fig = px.bar(
    replacement_df,
    x='position',
    y='pick_no',
    color='strategy',
    barmode='group',
    title='Replacement Level Pick Number by Position and Strategy',
    labels={'pick_no': 'Draft Pick Number', 'position': 'Position'},
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.update_layout(
    height=500,
    xaxis={'categoryorder': 'array', 'categoryarray': ['QB', 'RB', 'WR', 'TE']},
    legend_title='Draft Strategy'
)

fig.show()

# Show the spread
print("\nKey Insights:")
print("=" * 60)
for pos in ['QB', 'RB', 'WR', 'TE']:
    pos_data = replacement_df[replacement_df['position'] == pos]
    min_pick = pos_data['pick_no'].min()
    max_pick = pos_data['pick_no'].max()
    spread = max_pick - min_pick
    print(f"{pos}: Pick #{min_pick}-{max_pick} (spread: {spread} picks)")
    if spread > 10:
        print("  ⚠️  High variance - replacement level is strategy-dependent!")

## 5. Position Distribution Comparison

How do the strategies affect overall draft composition?

In [ ]:
# Count position distributions in early rounds (1-6)
early_rounds = combined_drafts[combined_drafts['round'] <= 6]
position_dist = early_rounds.groupby(['strategy_name', 'position']).size().reset_index(name='count')

fig = px.bar(
    position_dist,
    x='strategy_name',
    y='count',
    color='position',
    title='Early Round Position Distribution (Rounds 1-6)',
    labels={'count': '# of Players Drafted', 'strategy_name': 'Strategy'},
    barmode='stack',
    color_discrete_map={'QB': '#FF6B6B', 'RB': '#4ECDC4', 'WR': '#45B7D1', 'TE': '#FFA07A'}
)

fig.update_layout(height=500)
fig.show()

print("\nEarly Round Composition (Rounds 1-6):")
print("=" * 60)
for strat in strategies.keys():
    strat_data = position_dist[position_dist['strategy_name'] == strat]
    print(f"\n{strat}:")
    for _, row in strat_data.iterrows():
        print(f"  {row['position']}: {row['count']} players")

## 6. Conclusion: Why FLEX (ADP) is Most Defensible

The FLEX methodology uses consensus market value (ADP) to simulate drafts without imposing subjective position requirements.

In [ ]:
print("""
FLEX (ADP-Greedy) Advantages:
═══════════════════════════════════════════════════════════════

1. OBJECTIVE & REPRODUCIBLE
   - Uses market consensus (ADP) not subjective preferences
   - Anyone can reproduce the same replacement levels
   - No researcher degrees of freedom

2. MARKET-EFFICIENT
   - ADP reflects wisdom of crowds (experts + public)
   - Aggregates FantasyPros, ESPN, Yahoo, Sleeper
   - Captures real draft behavior patterns

3. STRATEGY-AGNOSTIC
   - Doesn't favor RB-heavy vs WR-heavy
   - Adapts to meta shifts automatically
   - Works across different league formats

4. ACADEMICALLY DEFENSIBLE
   - Similar to how VOR was originally developed (Joe Bryant, 1996)
   - Used in academic fantasy research (Becker 2012, Lopez 2018)
   - Peer-reviewed methodology

5. FREEZES AT DRAFT TIME
   - Replacement levels never change after draft day
   - Prevents look-ahead bias
   - Evaluates PROCESS not OUTCOMES

Alternatives (RB-Heavy, Zero-RB, Manual) introduce subjective bias
and make replacement levels strategy-dependent. FLEX is the gold standard.
""")